<a href="https://colab.research.google.com/github/keshavnath1/airflow-dags/blob/rag-pattern/GenAI_Financial_Mentor_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Finara – AI Financial Mentor (Colab Demo)
# -------------------------------------------------
# 1. Install any extra packages (none needed for core)
# -------------------------------------------------
# (The prototype is pure Python – no external deps)

# -------------------------------------------------
# 2. Paste the full prototype code
# -------------------------------------------------
"""
Finara — GenAI Financial Mentor Prototype
Implements all user stories from the product doc + What-If engine
"""

from datetime import datetime
import re
import json
from typing import Dict, List, Optional

class Goal:
    def __init__(self, goal_type: str, target_amount: float, current_amount: float = 0.0,
                 monthly_contribution: float = 0.0, target_date: Optional[str] = None):
        self.goal_type = goal_type
        self.target_amount = target_amount
        self.current_amount = current_amount
        self.monthly_contribution = monthly_contribution
        self.target_date = target_date

    def months_to_goal(self) -> float:
        if self.monthly_contribution <= 0:
            return float('inf')
        remaining = self.target_amount - self.current_amount
        return max(0, remaining / self.monthly_contribution)

    def progress_percent(self) -> float:
        return min(100.0, (self.current_amount / self.target_amount) * 100 if self.target_amount > 0 else 0)


class UserProfile:
    def __init__(self, name: str, annual_income: float, monthly_debts: float = 0.0,
                 current_savings: float = 0.0, credit_score: int = 700):
        self.name = name
        self.annual_income = annual_income
        self.monthly_income = annual_income / 12
        self.monthly_debts = monthly_debts
        self.current_savings = current_savings
        self.credit_score = credit_score
        self.goals: List[Goal] = []
        self.transactions: List[Dict] = []  # for demo budget

    def add_goal(self, goal: Goal):
        self.goals.append(goal)

    def calculate_house_affordability(self,
                                      down_payment_percent: float = 0.20,
                                      mortgage_rate: float = 0.068,
                                      term_years: int = 30,
                                      property_tax_rate: float = 0.012,
                                      insurance_rate: float = 0.004) -> Dict:
        max_total_payment = self.monthly_income * 0.43
        max_mortgage_payment = max_total_payment - self.monthly_debts

        if max_mortgage_payment <= 0:
            return {"error": "Income too low or debts too high to qualify for a mortgage"}

        monthly_rate = mortgage_rate / 12
        num_payments = term_years * 12

        if monthly_rate == 0:
            loan_amount = max_mortgage_payment * num_payments
        else:
            loan_amount = max_mortgage_payment * (1 - (1 + monthly_rate) ** -num_payments) / monthly_rate

        max_home_price = loan_amount / (1 - down_payment_percent)

        actual_down_payment = min(self.current_savings, max_home_price * down_payment_percent)
        actual_loan_amount = max_home_price - actual_down_payment

        actual_monthly_pi = actual_loan_amount * monthly_rate * (1 + monthly_rate)**num_payments / ((1 + monthly_rate)**num_payments - 1)
        monthly_tax_insurance = max_home_price * (property_tax_rate + insurance_rate) / 12
        total_monthly = actual_monthly_pi + monthly_tax_insurance

        return {
            "max_home_price": round(max_home_price, 2),
            "recommended_down_payment": round(max_home_price * down_payment_percent, 2),
            "actual_down_payment_possible": round(actual_down_payment, 2),
            "monthly_mortgage_pi": round(actual_monthly_pi, 2),
            "monthly_tax_insurance": round(monthly_tax_insurance,2),
            "total_monthly_payment": round(total_monthly,2),
            "dt_ratio": round(total_monthly / self.monthly_income * 100, 1)
        }

    def freelance_tax_retirement_recommendation(self, quarterly_gross: float, quarterly_expenses: float = 0.0):
        taxable_income_q = quarterly_gross - quarterly_expenses
        annual_projected = taxable_income_q * 4

        se_tax = taxable_income_q * 0.153
        if annual_projected <= 11600:
            income_tax_q = 0
        elif annual_projected <= 47150:
            income_tax_q = taxable_income_q * 0.12
        elif annual_projected <= 100525:
            income_tax_q = taxable_income_q * 0.22
        else:
            income_tax_q = taxable_income_q * 0.24

        total_tax_q = se_tax + income_tax_q
        safe_set_aside = total_tax_q * 1.25

        recommended_retirement = min(quarterly_gross * 0.20, 17500)

        return {
            "estimated_quarterly_tax": round(total_tax_q, 2),
            "safe_set_aside_amount": round(safe_set_aside, 2),
            "recommended_retirement_contribution": round(recommended_retirement, 2),
            "effective_tax_rate": round(total_tax_q / quarterly_gross * 100, 1)
        }

def categorize_transaction(merchant_name: str, amount: float) -> Dict:
    merchant = merchant_name.upper()
    rules = {
        "AMAZON|AMZ|AMZN": "Shopping",
        "WHOLE FOODS|TRADER JOE|KROGER|COSTCO|WALMART|TARGET": "Groceries",
        "UBER|LYFT|TAXI": "Transport",
        "STARBUCKS|COFFEE|CAFE|RESTAURANT| MCDONALDS": "Dining Out",
        "NETFLIX|SPOTIFY|HULU": "Entertainment",
        "CONED|VERIZON|COMCAST|ATT": "Bills & Utilities",
        "CVS|WALGREENS": "Health",
    }
    for pattern, category in rules.items():
        if re.search(pattern, merchant):
            confidence = 0.95 if 'AMZ' not in merchant else 0.92
            return {"category": category, "confidence": confidence}
    return {"category": "Other", "confidence": 0.65}


# -------------------------------------------------
# 3. Interactive UI (using ipywidgets)
# -------------------------------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output

# Global user object
user = None

def create_profile(b):
    global user
    name = name_in.value.strip() or "Anya"
    income = float(income_in.value)
    debts = float(debts_in.value or 0)
    savings = float(savings_in.value or 0)
    credit = int(credit_in.value) if credit_in.value else 700
    user = UserProfile(name, income, debts, savings, credit)
    clear_output()
    print(f"Profile created for **{user.name}**!")
    display(main_menu)

name_in = widgets.Text(placeholder="Your name")
income_in = widgets.FloatText(placeholder="Annual gross income")
debts_in = widgets.FloatText(placeholder="Monthly debt payments")
savings_in = widgets.FloatText(placeholder="Current savings")
credit_in = widgets.IntText(placeholder="Credit score (optional)")

profile_btn = widgets.Button(description="Create Profile", button_style='success')
profile_btn.on_click(create_profile)

profile_box = widgets.VBox([
    widgets.HTML("<h2>Create Your Financial Profile</h2>"),
    name_in, income_in, debts_in, savings_in, credit_in, profile_btn
])

# ---- Menu ----
def show_menu(*_):
    clear_output()
    display(main_menu)

house_btn = widgets.Button(description="House Affordability", icon='home')
goal_btn   = widgets.Button(description="Goals & What-If", icon='target')
freelance_btn = widgets.Button(description="Freelance Tax/Retirement", icon='briefcase')
cat_btn    = widgets.Button(description="Categorize Transaction", icon='tags')
budget_btn = widgets.Button(description="Budget Summary", icon='chart-bar')
exit_btn   = widgets.Button(description="Exit", button_style='danger', icon='power-off')

def on_house(_):
    clear_output()
    print("### House Affordability")
    res = user.calculate_house_affordability()
    if "error" in res:
        print(f"**Warning:** {res['error']}")
    else:
        for k, v in res.items():
            print(f"**{k.replace('_',' ').title()}**: ${v:,.2f}" if 'price' in k or 'payment' in k else f"**{k.replace('_',' ').title()}**: {v}")
    display(widgets.HBox([widgets.Button(description="Back", on_click=show_menu)]))
house_btn.on_click(on_house)

def on_goal(_):
    clear_output()
    print("### Your Goals")
    if not user.goals:
        print("No goals yet – let's add one!")
        gtype = widgets.Text(placeholder="Goal type")
        target = widgets.FloatText(placeholder="Target $")
        monthly = widgets.FloatText(placeholder="Monthly $")
        add = widgets.Button(description="Add Goal")
        def add_g(b):
            user.add_goal(Goal(gtype.value, target.value, user.current_savings if 'down' in gtype.value.lower() else 0, monthly.value))
            on_goal(None)
        add.on_click(add_g)
        display(widgets.VBox([gtype, target, monthly, add]))
    else:
        for i, g in enumerate(user.goals):
            m = g.months_to_goal()
            print(f"**{i+1}. {g.goal_type}** – $$ {g.current_amount:,.0f}/ $${g.target_amount:,.0f} ({g.progress_percent():.1f}%) – ~{m:.1f} months")
        # What-If
        whatif = widgets.FloatText(placeholder="Extra monthly $")
        run = widgets.Button(description="Run What-If")
        def run_w(b):
            extra = whatif.value
            idx = 0
            old = user.goals[idx].months_to_goal()
            user.goals[idx].monthly_contribution += extra
            new = user.goals[idx].months_to_goal()
            print(f"**Result:** +${extra:,.0f}/mo → saves **{old-new:.1f}** months!")
            user.goals[idx].monthly_contribution -= extra  # revert
        run.on_click(run_w)
        display(widgets.HBox([whatif, run]))
    display(widgets.HBox([widgets.Button(description="Back", on_click=show_menu)]))

goal_btn.on_click(on_goal)

def on_freelance(_):
    clear_output()
    print("### Freelance Tax & Retirement")
    gross = widgets.FloatText(placeholder="Quarterly gross")
    exp = widgets.FloatText(placeholder="Quarterly expenses (optional)")
    calc = widgets.Button(description="Calculate")
    def calc_f(b):
        res = user.freelance_tax_retirement_recommendation(gross.value, exp.value or 0)
        for k, v in res.items():
            print(f"**{k.replace('_',' ').title()}**: ${v:,.2f}" if isinstance(v, float) else f"**{k.replace('_',' ').title()}**: {v}")
    calc.on_click(calc_f)
    display(widgets.VBox([gross, exp, calc]))
    display(widgets.HBox([widgets.Button(description="Back", on_click=show_menu)]))

freelance_btn.on_click(on_freelance)

def on_cat(_):
    clear_output()
    print("### Transaction Categorizer")
    merch = widgets.Text(placeholder="Merchant name")
    amt   = widgets.FloatText(placeholder="Amount")
    go    = widgets.Button(description="Categorize")
    def go_c(b):
        c = categorize_transaction(merch.value, amt.value)
        print(f"**AI says:** {c['category']} (confidence {c['confidence']:.0%})")
    go.on_click(go_c)
    display(widgets.VBox([merch, amt, go]))
    display(widgets.HBox([widgets.Button(description="Back", on_click=show_menu)]))

cat_btn.on_click(on_cat)

def on_budget(_):
    clear_output()
    print("### Budget Summary")
    if not user.transactions:
        print("*(No transactions in this demo – add some in a real MVP via Plaid)*")
    else:
        # demo aggregation would go here
        pass
    display(widgets.HBox([widgets.Button(description="Back", on_click=show_menu)]))

budget_btn.on_click(on_budget)

def on_exit(_):
    clear_output()
    print("Thanks for trying **Finara**! Keep building your wealth.")
    # stop the notebook interaction
    import os, signal
    os.kill(os.getpid(), signal.SIGTERM)

exit_btn.on_click(on_exit)

main_menu = widgets.VBox([
    widgets.HTML("<h2>What would you like to do?</h2>"),
    house_btn, goal_btn, freelance_btn, cat_btn, budget_btn, exit_btn
])

# -------------------------------------------------
# 4. Start the app
# -------------------------------------------------
display(profile_box)

### House Affordability
**Warning:** Income too low or debts too high to qualify for a mortgage
